# GCAM electricity mixes in two EXIOBASE tables

Download *Current Policies* and *Net Zero 2050* for GCAM 6.0 NGFS in one MARIO call. The example shocks US generation in 2050 and retains the rest of the world as a second MRIO region.

Provide the hybrid EXIOBASE 3.3.18 (2011) SUT directory and the monetary EXIOBASE 3.8.2 (2015) SUT archive. Alternatively point the example at previously prepared baseline parquet directories. The two accounting years are distinct. Only generation shares are compared.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import mario

root = Path(os.environ.get("MARIO_IAM_DATA", "."))
market = "Electricity (generation)"
model = "GCAM 6.0 NGFS"
scenarios = ["Current Policies", "Net Zero 2050"]
iam = mario.download_ngfs(model=model, scenarios=scenarios)
region = model + "|USA"

## Prepare the generation market

Sum generation commodities into a common product and pool the foreign regions before cloning scenarios. Transmission, distribution and all activity classifications retain their original detail. Each baseline is prepared once.

In [ ]:
def baseline(unit):
    cached = root / unit / "flows"
    if cached.is_dir():
        return mario.parse_from_parquet(path=str(cached), table="SUT", mode="flows")
    path = os.environ["EXIOBASE_" + unit.upper() + "_SUT"]
    options = {"extensions": []} if unit == "Hybrid" else {}
    db = mario.parse_exiobase(table="SUT", unit=unit, path=path, **options)
    regions = db.get_index("Region")
    labels = db.get_index("Commodity", flat=True)
    products = {c: market if c.lower().startswith("electricity by ") or c.lower() == "electricity nec"
                else c for c in labels}
    db.aggregate(io={"Commodity": pd.DataFrame({"Aggregation": products})},
                 levels=["Region", "Commodity"],
                 region_aggregation={"US": ["US"], "Rest of world": [r for r in regions if r != "US"]})
    return db

physical_db = baseline("Hybrid")
monetary_db = baseline("Monetary")

## Apply and verify both scenarios

Relative technology prices follow Peng et al., Supporting Information 1, Table S6 ([article](https://doi.org/10.1021/acs.est.6c10592)). A common multiplier can anchor the prices to GCAM's regional mean. Only their ratios affect the monetary shares. Under a common producer price assumption, quantity and value shares would coincide. The output shows shares within the twelve generation activities.

In [ ]:
relative = {"coal": 2.76, "gas": 1.47, "nuclear": 4.08, "hydro": 1.20, "wind": 1.00,
            "petroleum and other oil derivatives": 3.99, "biomass and waste": 1.79,
            "Geothermal": 2.07, "solar photovoltaic": 0.95, "solar thermal": 3.55}
ratios = pd.Series({"Production of electricity by " + k: v for k, v in relative.items()})
results = []
for scenario in scenarios:
    select = dict(model=model, scenario=scenario, year=2050, regions={"US": region}, unit="EJ/yr")
    hybrid_case = physical_db.shock_from_iam(iam, **select)
    money_case = monetary_db.shock_from_iam(iam, **select, prices="peng-2026")
    # Independently verify the solved flows against the physical mix and the price ratios.
    physical = iam.supply_mix(**select, basis="physical")
    quantities = pd.Series(physical["US"])
    price_row = iam.frame.query("Model == @model and Scenario == @scenario and Region == @region")
    quoted = price_row.loc[(price_row.Variable == "Price|Secondary Energy|Electricity")
                           & (price_row.Unit == "US$2010/GJ"), 2050].item()
    prices = ratios * quoted / quantities.loc[ratios.index].dot(ratios)
    monetary = iam.supply_mix(**select, basis="value", prices={"US": prices})
    for db, case, basis, target in [(physical_db, hybrid_case, "physical", physical),
                                    (monetary_db, money_case, "value", monetary)]:
        supply = db.query("S", scenarios=case)[("US", "Commodity", market)]
        flows = supply.xs(("US", "Activity")).loc[quantities.index]
        shares = flows / flows.sum()
        np.testing.assert_allclose(shares, pd.Series(target["US"]), atol=1e-9)
        recovered = flows.copy()
        if basis == "value":
            recovered.loc[prices.index] /= prices
        np.testing.assert_allclose(recovered / recovered.sum(), quantities, atol=1e-9)
        results.append(shares.rename((scenario, basis)))
pd.concat(results, axis=1).mul(100).round(2)

## Use custom scenario and variable labels

The following small, illustrative steel export defines two technology routes in Mt/year. Supply a mapping to activities present in your steel database. For a study workbook with several worksheets, pass the worksheet name to `read_iamc`. For example, the Ghezzi et al. supplement uses `sheet_name="8. Results | GCAM full data"` and the scenario name `NDC_LTT_CBAM` ([workbook](https://doi.org/10.1021/acs.est.5c15099.s001)).

In [ ]:
custom = pd.DataFrame({
    "Model": ["GCAM-custom"] * 3, "Scenario": ["Regional steel policy"] * 3,
    "Region": ["North"] * 3, "Unit": ["Mt/yr"] * 3,
    "Variable": ["Steel|Scrap EAF", "Steel|Blast furnace", "Steel"], 2050: [30, 70, 100],
})
steel = mario.read_iamc(custom).supply_mix(
    model="GCAM-custom", scenario="Regional steel policy", year=2050, regions={"US": "North"},
    unit="Mt/yr", basis="physical", variables={"Scrap-fed EAF": "Steel|Scrap EAF",
        "Blast furnace steel": "Steel|Blast furnace"}, total="Steel")
assert steel == {"US": {"Scrap-fed EAF": 0.3, "Blast furnace steel": 0.7}}
steel

### Ghezzi et al. steelmaking routes

To run the published case, set `GHEZZI_SUPPLEMENT` to the [study workbook](https://doi.org/10.1021/acs.est.5c15099.s001). The figure 3–4 sheet lists production by route in a separate technology column. Four scrap-EAF finishing variants share the display label `EAF_scrap`. Sum their quantities before forming shares. Check the route total against the figure 2 production total, reported to one decimal place. That sheet labels the same policy scenario `CBAM`.

In [ ]:
study_file = os.environ.get("GHEZZI_SUPPLEMENT")
if study_file:
    routes = pd.read_excel(study_file, sheet_name="4. Results | Figure3&4", usecols="A:I")
    routes = routes.loc[routes.Variable == "Steel production"].copy()
    routes["Variable"] = "Steel output|" + routes["Steel technology"]
    keys = ["Model", "Scenario", "Region", "Variable", "Unit", "Year"]
    annual = routes.groupby(keys, as_index=False)["Value"].sum()
    custom_vars = {v.split("|", 1)[1]: v for v in annual.Variable.unique()}
    real_mix = mario.read_iamc(annual).supply_mix(
        model="GCAM 7.0", scenario="NDC_LTT_CBAM", year=2050, regions={"EU27": "EU27"},
        variables=custom_vars, unit="Mton steel", basis="physical")
    production = annual.query("Scenario == 'NDC_LTT_CBAM' and Region == 'EU27' and Year == 2050").Value.sum()
    totals = pd.read_excel(study_file, sheet_name="3. Results | Figure2", usecols="A:G")
    reference = totals.query("Scenario == 'CBAM' and Region == 'EU27' and Year == 2050 and Variable == 'Steel production'").Value.item()
    np.testing.assert_allclose(production, reference, atol=0.05, rtol=0)
    print("EU27 scrap-EAF share:", round(real_mix["EU27"]["EAF_scrap"] * 100, 2), "%")
